In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna

In [3]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [4]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import kagglehub
from pathlib import Path

path = kagglehub.dataset_download("parsalatifi/s-and-p-500-us-stock-market-data")



DATASET_PATH = Path(path)
df = pd.read_csv(DATASET_PATH / "master_stock_data.csv")

In [6]:
df=pd.read_csv('C:/Users/prath/.cache/kagglehub/datasets/parsalatifi/s-and-p-500-us-stock-market-data/versions/3/master_stock_data.csv')
df.head(10)

,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,1999-11-18,A,26.189770,31.473534,0.0,35.765381,28.612303,32.546494,0.0,62546380
1,1999-11-19,A,24.032093,28.880545,0.0,30.758226,28.478184,30.713518,0.0,15234146
2,1999-11-22,A,26.189770,31.473534,0.0,31.473534,28.657009,29.551144,0.0,6577870
3,1999-11-23,A,23.808886,28.612303,0.0,31.205294,28.612303,30.400572,0.0,5975611
4,1999-11-24,A,24.441307,29.372318,0.0,29.998213,28.612303,28.701717,0.0,4843231
5,1999-11-26,A,24.515709,29.461731,0.0,29.685265,29.148785,29.238197,0.0,1729466
6,1999-11-29,A,25.073727,30.132332,0.0,30.355865,29.014664,29.327612,0.0,4074751
7,1999-11-30,A,25.110933,30.177038,0.0,30.713518,29.282904,30.042917,0.0,4310034
8,1999-12-01,A,25.557346,30.713518,0.0,31.071173,29.953505,30.177038,0.0,2957329
9,1999-12-02,A,26.264172,31.562946,0.0,32.188843,30.892345,31.294706,0.0,3069868


In [12]:
df.drop(columns=["Ticker","Dividends","Stock Splits","Adj Close","Date"],inplace=True)

In [13]:
print(df.isna().sum())

Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [14]:
df = df.dropna()

In [15]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=["Close"]),df.iloc[:,:1],test_size=0.2)

In [16]:
x_scaler=StandardScaler()
x_train=x_scaler.fit_transform(x_train)
x_test=x_scaler.transform(x_test)

In [17]:
y_scaler=StandardScaler()
y_train=y_scaler.fit_transform(y_train)
y_test=y_scaler.transform(y_test)

In [18]:
#create CustomDataset Class
class CustomDataset(Dataset):
  def __init__(self, x, y):

    self.x = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.float32)

  def __len__(self):

    return len(self.x)

  def __getitem__(self, index):

    return self.x[index], self.y[index]


In [19]:
train_dataset = CustomDataset(x_train, y_train)

In [20]:
test_dataset = CustomDataset(x_test, y_test)

In [21]:
train_dataset[:10]

(tensor([[-0.4322, -0.4324, -0.4319,  0.0575],
         [ 0.9567,  0.9520,  0.9706, -0.1503],
         [-0.3043, -0.3081, -0.3057, -0.1785],
         [-0.4413, -0.4415, -0.4408,  0.0789],
         [ 0.2224,  0.2282,  0.2279,  0.1364],
         [-0.2866, -0.3233, -0.2869, -0.0203],
         [ 0.1751,  0.1624,  0.1595, -0.1469],
         [ 2.3687,  2.3904,  2.3623, -0.1408],
         [-0.0346, -0.0280, -0.0305, -0.1698],
         [-0.3568, -0.3578, -0.3570,  0.0457]]),
 tensor([[-0.4326],
         [ 0.9360],
         [-0.3084],
         [-0.4410],
         [ 0.2233],
         [-0.3229],
         [ 0.1787],
         [ 2.3829],
         [-0.0319],
         [-0.3563]]))

In [32]:
class MyNN(nn.Module):
  def __init__(self,input_dim,output_dim,num_hidden_layers,neurons_per_layer,dropout_rate):
    super().__init__()
    layers=[]

    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim=neurons_per_layer

    layers.append(nn.Linear(neurons_per_layer,output_dim))

    self.model=nn.Sequential(*layers)

    
  def forward(self,x):

      return self.model(x)

In [35]:
# objective function

def objective(trial):
  # next hyperparameter values from the search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
  neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
  epochs = trial.suggest_int("epochs", 10, 40, step=5)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=True)

  # model init
  input_dim = 4
  output_dim = 1

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # optimizer selection
  criterion = nn.MSELoss()
#   optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

  if optimizer_name == 'Adam':
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop

  for epoch in range(epochs):
    model.train()
    for batch_features, batch_labels in train_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # calculate loss
      loss = criterion(outputs.squeeze(), batch_labels.squeeze())

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update grads
      optimizer.step()


  # evaluation
  model.eval()
  # evaluation on test data
  preds, targets = [], []

  with torch.no_grad():
      for x_test_batch, y_test_batch in test_loader:
          x_test_batch = x_test_batch.to(device)
          outputs = model(x_test_batch)
          preds.append(outputs.cpu())
          targets.append(y_test_batch)

  y_pred = torch.cat(preds).numpy().ravel()
  y_true = torch.cat(targets).numpy().ravel()

  r2   = r2_score(y_true, y_pred)

  return r2


In [36]:
study = optuna.create_study(direction='maximize')

[I 2026-10-04 13:48:44,621] A new study created in memory with name: no-name-6e5e7951-b113-4c02-8b63-5f16ee2c9d92


In [38]:
study.optimize(objective, n_trials=20)

[I 2026-10-04 15:53:34,159] Trial 2 finished with value: 0.4077916741371155 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 16, 'epochs': 40, 'learning_rate': 0.004015480877059604, 'dropout_rate': 0.2, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 0.00014656886752798735}. Best is trial 1 with value: 0.8876959681510925.
[I 2026-10-04 16:35:27,438] Trial 3 finished with value: 0.5473811626434326 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 104, 'epochs': 15, 'learning_rate': 0.008067664521958246, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'SGD', 'weight_decay': 0.00014729096773921118}. Best is trial 1 with value: 0.8876959681510925.
[I 2026-10-04 18:07:51,260] Trial 4 finished with value: 0.6371356248855591 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 40, 'epochs': 35, 'learning_rate': 1.0306652224443996e-05, 'dropout_rate': 0.4, 'batch_size': 32, 'optimizer': 'RMSprop', 'weight_decay': 2.2783348471062375e-05}. Best is tri

In [39]:
study.best_value

0.9967198967933655

In [40]:
study.best_params

{'num_hidden_layers': 2,
 'neurons_per_layer': 24,
 'epochs': 15,
 'learning_rate': 1.2392889074215176e-05,
 'dropout_rate': 0.2,
 'batch_size': 128,
 'optimizer': 'SGD',
 'weight_decay': 1.145060732085222e-05}